In [40]:
import talib
import scipy.stats
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.subplots as sub
import quantstats as qs
from jup.helpers.charts import (
    get_layout,
    get_price_trace,
    add_trade_markers,
    remove_x_gaps,
    detect_range_rule,
)
from jup.helpers.data_sources import get_exante_data, resample_ohlc

pd.options.display.width = 150

# ticker = "EUR:CHF"
# ticker = "AUD:NZD"
# ticker = "GBP:CHF"
ticker = "EMQQ.ARCA"  # 117T

src = get_exante_data(ticker, start="2018-01-01 00:00+0", end="2021-06-01 00:00+0")
# src = resample_ohlc(src, "3h")
# _, range_rule = detect_range_rule(src)

# print(f"{ticker}, {len(src)} × {range_rule}")
print(f"From: {src.index[0]:%Y-%m-%d %H:%M}")
print(f"To:   {src.index[-1]:%Y-%m-%d %H:%M}")

From: 2018-01-02 12:00
To:   2021-05-28 20:00


In [41]:
md = src.copy(deep=True)

md["count"] = 1
md = md.resample("1d").apply({
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum",
    "count": "size",
})
md.dropna(inplace=True)
md["zero"] = 0

open, high, low, close, volume = md.open, md.high, md.low, md.close, md.volume

md["cnt_per_change"] = (md["close"] - md["open"]) / md["count"]
md["cnt_per_change"] = md["cnt_per_change"]

md["volume"] = talib.EMA(md["volume"] / 2000, timeperiod=10)

md["cnt_per_change"] = talib.SMA(md["cnt_per_change"], timeperiod=30)

md["mfi"] = talib.MFI(high, low, close, volume, timeperiod=20)
# md["ind"] = talib.TRIX(close, timeperiod=10).abs()
# md["ind"] = talib.BOP(md["open"], md["high"], md["low"], md["close"])

md["ind"] = talib.EMA(md["count"], timeperiod=50) - talib.EMA(md["count"], timeperiod=10)

# real = MFI(high, low, close, volume, timeperiod=14)

md["count"] = talib.EMA(md["count"], timeperiod=10)

# md.tail(5)


layout = get_layout()

fig = sub.make_subplots(
    figure=go.Figure(layout=layout),
    rows=6,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02,
    row_heights=[0.5, 0.2, 0.2, 0.2, 0.2, 0.2],
)
# Правильный rangeslider
# fig.add_trace(go.Scatter(x=df.index, y=df["mid"]), row=3, col=1)
# fig.update_layout(xaxis3=dict(rangeslider=dict(visible=True, thickness=0.05)))
# fig['layout']['xaxis2']['visible'] = False
# fig['layout']['yaxis3']['visible'] = False

line = dict(color="rgba(0,0,0,0.2)", width=1)
line_or = dict(color="#f50", width=1)


fig.add_trace(go.Scatter(x=md.index, y=md["close"], line=line, name="close"))

fig.add_trace(go.Scatter(x=md.index, y=md["volume"]), row=2, col=1)
fig.add_trace(go.Scatter(x=md.index, y=md["mfi"]), row=3, col=1)
fig.add_trace(go.Scatter(x=md.index, y=md["count"]), row=2, col=1)
fig.add_trace(go.Scatter(x=md.index, y=md["cnt_per_change"]), row=6, col=1)

fig.add_trace(go.Scatter(x=md.index, y=md["ind"]), row=4, col=1)

# color = np.where((md["ind"] < 0.2), 'red', '#0b0')
# fig.add_trace(go.Bar(x=md.index, y=md["ind"], marker_color=color), row=4, col=1)


# remove_x_gaps(fig, md, md, skip_start=0, skip_end=0)


fig.add_trace(go.Scatter(x=md.index, y=md["zero"]+50), row=3, col=1)
fig.add_trace(go.Scatter(x=md.index, y=md["zero"]), row=4, col=1)
fig.add_trace(go.Scatter(x=md.index, y=md["zero"]), row=6, col=1)


# Индикаторы на графике
# line = dict(color="rgba(0,0,0,0.2)", width=1)
# line_or = dict(color="#f50", width=1)
# fig.add_trace(go.Scatter(x=df.index, y=df["top"], line=line, name="top"))
# fig.add_trace(go.Scatter(x=df.index, y=df["mid"], line=line, name="mid"))
# fig.add_trace(go.Scatter(x=df.index, y=df["bottom"], line=line, name="btm"))

